# Figure 3A,B — In-silico saturation mutagenesis at representative meQTLs

**This notebook loads the Melody-MT model and is meant to run on a GPU machine** that has the
Melody model code, the 39-track checkpoint, and the GRCh38 primary-assembly FASTA. Set the
three paths in the configuration cell below (it needs `selene-sdk` and `tangermeme` installed).
It reproduces the full Figure 3A/B panels — the predicted methylation profile before/after the
SNP **and** the in-silico saturation-mutagenesis sequence logo, computed from the Melody model.

- **3A** — MDSs index 64, chr10:132,913,360 **A->G**: creates an IRF motif -> methylation decrease.
- **3B** — MDSs index 397, chr5:151,239,432 **C->T**: disrupts a CTCF motif -> methylation increase.

Blood tracks (22-26) are averaged; the methylation profile is shown over +/-1500 bp around the
SNP and the ISM logo over +/-75 bp.

In [ ]:
import warnings; warnings.filterwarnings('ignore')
from loguru import logger as _lg; _lg.remove()
import sys, os
import matplotlib.pyplot as plt

# ===== Configure these three paths for your environment =====
CODE_REPO  = '/path/to/Melody'                 # a checkout of the Melody model code repository
CHECKPOINT = '/path/to/Melody-MT-39.pth'       # 39-track Melody-MT checkpoint
GENOME_FA  = '/path/to/GRCh38.primary_assembly.fa'
# ============================================================
# Requires: pip install selene-sdk tangermeme torch

sys.path.insert(0, CODE_REPO)          # the Melody model code (models, stateless, eqtl_pure_util, ...)
from selene_sdk.sequences import Genome
from models import Melody
from stateless import load_ckpt
from ism_plotting import plot_meqtl_plus_ism      # helper file next to this notebook

genome = Genome(input_path=GENOME_FA, blacklist_regions='hg38')
genome.get = genome.get_encoding_from_coords      # (L, 4) one-hot accessor the model was trained on
model = Melody(None, n_track=39)
load_ckpt(model, CHECKPOINT)
model = model.cuda()
TRACKS = [22, 23, 24, 25, 26]          # blood cell types (averaged)

## Figure 3A — chr10 A→G (IRF motif gain → methylation decrease)

In [ ]:
fig = plot_meqtl_plus_ism(
    dataset_name='MDSs', idx=64, model=model, genome=genome,
    track_indices=TRACKS, half_model_input_len=5000, margin=0,
    view_bp=75, ism_flank=75, batch_size=128, landscape_half_len=1500)
plt.show()

## Figure 3B — chr5 C→T (CTCF motif disruption → methylation increase)

In [ ]:
fig = plot_meqtl_plus_ism(
    dataset_name='MDSs', idx=397, model=model, genome=genome,
    track_indices=TRACKS, half_model_input_len=5000, margin=0,
    view_bp=75, ism_flank=75, batch_size=128, landscape_half_len=1500)
plt.show()